# Mini-Übung: CRF-Sequenzbewertung

Nutze dieselben festen Parameter wie im Beispiel für die Sequenz `Clara arbeitet in Hamburg`. Ziel ist nicht das Training, sondern das sichere Verständnis von Formen, Score und Viterbi.

In [ ]:
import itertools
import numpy as np
import pandas as pd

labels=np.array(["PER","LOC","O"])
label_id={label:i for i,label in enumerate(labels)}
feature_namen=["gross", "nach_in", "verb", "ist_in", "bias"]

# Zeilen: PER, LOC, O; Spalten: die fünf Features
W=np.array([
    [ 1.4, -1.0, -1.0, -0.5,  0.0],
    [ 1.0,  1.8, -1.0, -0.5, -0.2],
    [-0.8, -0.4,  1.2,  1.4,  0.5],
])

# Zeile = vorheriges Label, Spalte = aktuelles Label
A=np.array([
    [ 0.4, -0.6, 0.7],
    [-0.5,  0.3, 0.6],
    [ 0.2,  1.1, 0.8],
])

def features(tokens):
    result=[]
    for t,token in enumerate(tokens):
        result.append([
            float(token[0].isupper()),
            float(t>0 and tokens[t-1].lower()=="in"),
            float(token.lower() in {"lebt","wohnt","arbeitet"}),
            float(token.lower()=="in"),
            1.0,
        ])
    return np.array(result)

def emissions(tokens):
    F=features(tokens)
    return F@W.T

def sequenz_score(E,sequenz,A_matrix=A):
    ids=[label_id[x] for x in sequenz]
    lokal=sum(E[t,k] for t,k in enumerate(ids))
    transition=sum(A_matrix[ids[t-1],ids[t]] for t in range(1,len(ids)))
    return lokal+transition

def logsumexp(x,axis=None):
    maximum=np.max(x,axis=axis,keepdims=True)
    wert=maximum+np.log(np.sum(np.exp(x-maximum),axis=axis,keepdims=True))
    return np.squeeze(wert,axis=axis) if axis is not None else float(wert.squeeze())

## Aufgabe 1 - Feature- und Score-Matrix

Erzeuge `F` und `E`. Gib beide als DataFrame aus und notiere die Formen von `F`, `W`, `A` und `E`. Warum hängt keine Parameterform von der Sequenzlänge ab?

In [ ]:
tokens=["Clara","arbeitet","in","Hamburg"]
# Dein Code

## Aufgabe 2 - Zwei Kandidaten vergleichen

Berechne die Scores für

- `PER O O LOC`
- `LOC O O PER`

Zerlege mindestens den ersten Score in lokale und Übergangsbeiträge. Welcher Kandidat ist plausibler und warum?

In [ ]:
# Dein Code

## Aufgabe 3 - Viterbi ergänzen

Vervollständige die markierten Stellen. Vergleiche Ergebnis und Score mit der besten Sequenz aus vollständiger Aufzählung.

In [ ]:
def viterbi_uebung(E,A_matrix=A):
    delta=E[0].copy()
    rueckzeiger=[]
    for t in range(1,len(E)):
        kandidaten=...  # vorherige Scores plus alle Übergänge
        rueckzeiger.append(...)
        delta=...       # lokale Scores plus bester Vorgänger
    pfad=[int(np.argmax(delta))]
    for bp in reversed(rueckzeiger):
        pfad.append(...)
    pfad=pfad[::-1]
    return labels[pfad].tolist(),float(np.max(delta))

# Dein Test

## Aufgabe 4 - Übergänge verändern

Kopiere `A` und ändere `A[O, LOC]` von `1.1` auf `-3.0`. Führe Viterbi erneut aus.

1. Ändert sich die beste Sequenz?
2. Was zeigt das über lokale Token-Scores und globale Sequenzstruktur?
3. Worin unterscheidet sich Forward von Viterbi?

In [ ]:
# Dein Code und deine Antworten